# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


```
prompt | llm | parser
```

In [13]:
import sys
sys.path.insert(0, "..")

from common_config import llm_connect

llm = llm_connect(
    model="gpt-5.4-mini",
    temperature=0,
    max_tokens=2086,
)

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


## 1. 기본 체인: prompt | llm | parser

In [15]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | 연산자로 체인 구성
chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 데이터베이스"})
print(result)

벡터 데이터베이스는 **문서, 이미지, 음성 같은 데이터를 숫자 벡터로 저장해 비슷한 의미의 데이터를 빠르게 찾아주는 데이터베이스**입니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [16]:
for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩은 **텍스트나 이미지 같은 데이터를 컴퓨터가 이해하고 비교할 수 있도록 숫자 벡터로 바꾸는 표현 방식**입니다.

In [17]:
for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)

임베딩은 **텍스트, 이미지 같은 데이터를 컴퓨터가 이해할 수 있도록 의미가 비슷한 것끼리 가까워지도록 숫자 벡터로 바꾸는 방법**입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [18]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [19]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}



```mermaid
flowchart TB
    subgraph S1["① RunnablePassthrough() : 그대로 통과"]
        direction LR
        A1["입력<br/>{'topic': 'RAG'}"] --> B1["RunnablePassthrough()"] --> C1["출력<br/>{'topic': 'RAG'}"]
    end

    subgraph S2["② .assign() : 입력 유지 + 새 키 추가"]
        direction LR
        A2["입력<br/>{'word': 'hello'}"] --> P2["기존 값 유지<br/>word: 'hello'"]
        A2 --> F2["upper 함수<br/>x['word'].upper()"]
        P2 --> M2(("병합"))
        F2 -->|"upper: 'HELLO'"| M2
        M2 --> C2["출력<br/>{'word': 'hello',<br/>'upper': 'HELLO'}"]
    end

    subgraph S3["③ 실전 RAG 패턴"]
        direction LR
        A3["입력<br/>{'question': ...}"] --> B3[".assign(context=retriever)"]
        B3 --> D3["{'question', 'context'}"] --> PR["prompt"] --> L["llm"] --> O["parser"] --> R["최종 답변"]
    end

    S1 ~~~ S2 ~~~ S3

    classDef input fill:#dbeafe,stroke:#2563eb,color:#1e3a8a
    classDef pass fill:#fef3c7,stroke:#d97706,color:#78350f
    classDef func fill:#fce7f3,stroke:#db2777,color:#831843
    classDef output fill:#dcfce7,stroke:#16a34a,color:#14532d
    class A1,A2,A3 input
    class B1,P2,B3,M2 pass
    class F2,PR,L,O func
    class C1,C2,D3,R output
```

> 🟦 입력 · 🟨 Passthrough 처리 · 🟪 함수/체인 실행 · 🟩 출력


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [20]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 **LLM(대규모 언어 모델) 기반 애플리케이션을 “그래프” 구조로 만들고 실행하는 프레임워크**입니다.  
특히 **상태(state)를 유지하면서, 여러 단계의 작업을 조건 분기/반복/루프로 제어해야 하는 AI 워크플로우**에 잘 맞습니다.

### 쉽게 말하면
- **LangChain**이 주로 “LLM + 도구 + 체인”을 쉽게 연결하는 데 강하다면,
- **LangGraph**는 그걸 더 발전시켜서 **복잡한 에이전트 흐름을 노드와 엣지로 표현**하게 해줍니다.

### 왜 쓰나요?
일반적인 LLM 앱은 한 번 질문 → 한 번 답변이면 끝나지만, 실제 AI 시스템은 보통:
- 계획 세우기
- 도구 호출
- 결과 확인
- 다시 시도
- 사람 승인 받기
- 조건에 따라 다른 경로로 이동

같은 **여러 단계의 흐름**이 필요합니다.  
LangGraph는 이런 흐름을 **그래프 형태로 명시적으로 구성**할 수 있게 해줍니다.

### 핵심 개념
- **Node(노드)**: 한 단계의 작업(예: LLM 호출, 검색, 요약)
- **Edge(엣지)**: 다음 단계로의 이동
- **State(상태)**: 단계들 사이에 공유되는 데이터
- **Conditional routing**: 결과에 따라 다음 노드를 다르게 선택
- **Loops/recursion**: 필요하면 반복 실행 가능

### 어떤 데 유용한가요?
- 멀티스텝 에이전트
- RAG 파이프라인
- 사람 검토가 들어가는 업무 자동화
- 실패 시 재시도 로직
- 분기/반복이 많은 복잡한 워크플로우

### 한 줄 요약
**LangGraph는 LLM 앱을 “상태를 가진 그래프 워크플로우”로 만들게 해주는 프레임워크입니다.**

원하시면 제가 **LangChain과 LangGraph 차이**를 표로 비교해드리거나, **간단한 예제 코드**로도 설명해드릴게요.


```mermaid
flowchart TB
    subgraph S1["① 일반 함수 → Runnable로 감싸기"]
        direction LR
        F1["def add_prefix(text)<br/>(일반 파이썬 함수)"] --> W1["RunnableLambda(add_prefix)"] --> R1["Runnable<br/>(invoke / batch / stream 가능)"]
    end

    subgraph S2["② 체인 마지막에 후처리로 연결"]
        direction LR
        A2["입력<br/>{'topic': 'LangGraph'}"] --> PR["prompt"] --> L["llm"] --> O["StrOutputParser"]
        O -->|"'LangGraph는 ...'"| W2["RunnableLambda<br/>(add_prefix)"]
        W2 --> C2["출력<br/>'[결과] LangGraph는 ...'"]
    end

    S1 ~~~ S2

    classDef input fill:#dbeafe,stroke:#2563eb,color:#1e3a8a
    classDef pass fill:#fef3c7,stroke:#d97706,color:#78350f
    classDef func fill:#fce7f3,stroke:#db2777,color:#831843
    classDef output fill:#dcfce7,stroke:#16a34a,color:#14532d
    class F1,A2 input
    class W1,W2 pass
    class PR,L,O func
    class R1,C2 output
```

> 🟦 입력 · 🟨 Runnable 처리 · 🟪 함수/체인 실행 · 🟩 출력


## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [21]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] RAG는 **Retrieval-Augmented Generation**의 줄임말로, 한국어로는 보통 **검색 증강 생성**이라고 번역합니다.

쉽게 말하면:

- **검색(Retrieval)**: 질문과 관련된 정보를 외부 문서, 데이터베이스, 웹 등에서 찾아오고
- **증강(Augmented)**: 찾아온 정보를 바탕으로
- **생성(Generation)**: 그 내용을 이용해 답변을 만들어내는 방식입니다

## 왜 쓰나요?
일반적인 AI 모델은 **학습할 때 본 지식**에 의존합니다.  
그래서 최신 정보나 회사 내부 문서 같은 건 잘 모를 수 있어요.

RAG를 쓰면:
- 최신 정보 반영이 쉽고
- 내부 문서/매뉴얼 기반 답변이 가능하고
- 모델이 헛소리하는 걸 줄이는 데 도움이 됩니다

## 예시
예를 들어 사용자가  
“우리 회사 휴가 규정이 뭐야?”  
라고 물으면,

1. 시스템이 사내 규정 문서를 검색하고
2. 관련 조항을 찾은 뒤
3. 그 내용을 바탕으로 AI가 자연스럽게 답변합니다

즉, **AI가 그냥 기억만으로 답하는 게 아니라, 필요한 자료를 찾아보고 답하는 구조**입니다.

원하시면 제가 이어서  
- **RAG의 작동 과정**
- **장점과 한계**
- **LLM과 RAG의 차이**
도 한국어로 쉽게 설명해드릴게요.
[English] RAG stands for **Retrieval-Augmented Generation**.

In simple English, it means:

1. **Retrieve**: The system first looks up relevant information from a source, like documents, a database, or the web.
2. **Augment**: It adds that information to the prompt or context.
3. **Generate**: Then the AI model uses both the user’s question an

```mermaid
flowchart LR
    A["입력<br/>{'topic': 'RAG'}"] --> P{{"RunnableParallel<br/>(같은 입력을 동시에 분배)"}}

    subgraph K["korean = chain_ko"]
        direction LR
        PK["prompt_ko"] --> LK["llm"] --> OK["parser"]
    end

    subgraph E["english = chain_en"]
        direction LR
        PE["prompt_en"] --> LE["llm"] --> OE["parser"]
    end

    P --> PK
    P --> PE
    OK -->|"korean"| M(("결과 합치기"))
    OE -->|"english"| M
    M --> C["출력<br/>{'korean': '...',<br/>'english': '...'}"]

    classDef input fill:#dbeafe,stroke:#2563eb,color:#1e3a8a
    classDef pass fill:#fef3c7,stroke:#d97706,color:#78350f
    classDef func fill:#fce7f3,stroke:#db2777,color:#831843
    classDef output fill:#dcfce7,stroke:#16a34a,color:#14532d
    class A input
    class P,M pass
    class PK,LK,OK,PE,LE,OE func
    class C output
```

> 🟦 입력 · 🟨 Runnable 처리 · 🟪 체인 실행 · 🟩 출력  
> 두 체인이 **동시에** 실행되므로, 순차 실행보다 전체 대기 시간이 짧아짐


## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [22]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

NameError: name 'ChatOpenAI' is not defined